# 01 — Data Understanding & Cleaning (NusaMart / Olist)

**Tujuan notebook ini:** mengubah 5 file CSV mentah Olist menjadi 4 tabel bersih (star schema) yang siap dimasukkan ke PostgreSQL.

```
data/raw/ (5 CSV Olist)  →  notebook ini  →  data/processed/ (fact_sales, dim_customer, dim_product, dim_date)
```

**Aturan penting:** setiap keputusan cleaning (misalnya membuang data) harus punya alasan, dan alasannya dicatat di sel *"Catatan keputusan"*. Keputusan inilah yang nanti ditanya interviewer.

**Sebelum menjalankan:** taruh 5 file berikut di folder `data/raw/`:
`olist_orders_dataset.csv`, `olist_order_items_dataset.csv`, `olist_customers_dataset.csv`, `olist_products_dataset.csv`, `product_category_name_translation.csv`

## 1. Load data
Membaca 5 tabel yang kita pakai. Tabel Olist lain (payments, reviews, sellers, geolocation) sengaja tidak dipakai karena di luar scope.

In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_tr": "product_category_name_translation.csv",
}

raw = {name: pd.read_csv(RAW_DIR / fname) for name, fname in FILES.items()}

for name, df in raw.items():
    print(f"{name:12s} {df.shape[0]:>8,} baris  {df.shape[1]:>3} kolom")

orders         99,441 baris    8 kolom
items         112,650 baris    7 kolom
customers      99,441 baris    5 kolom
products       32,951 baris    9 kolom
category_tr        71 baris    2 kolom


## 2. Kenali struktur data
Untuk setiap tabel: tipe data tiap kolom dan persentase nilai kosong (missing). Kolom yang banyak kosong perlu diputuskan: dibuang, diisi, atau dibiarkan.

In [2]:
for name, df in raw.items():
    print(f"\n===== {name} =====")
    print(df.dtypes.to_string())
    missing_pct = (df.isna().mean() * 100).round(2)
    missing_pct = missing_pct[missing_pct > 0]
    print("\nMissing (%):", "tidak ada" if missing_pct.empty else "")
    if not missing_pct.empty:
        print(missing_pct.to_string())


===== orders =====
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object

Missing (%): 
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98

===== items =====
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64

Missing (%): tidak ada

===== customers =====
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object

Missing (%): tidak ada

===== products =====
product_id                 

## 3. Cek duplikat & primary key
Primary key = kolom yang seharusnya unik per baris. Jika ada duplikat, penghitungan revenue bisa dobel.

In [3]:
def cek_unik(df, cols, nama):
    dup = int(df.duplicated(subset=cols).sum())
    status = "OK" if dup == 0 else "PERLU DICEK"
    print(f"{nama:12s} key={cols}: {dup} duplikat  -> {status}")

cek_unik(raw["orders"], ["order_id"], "orders")
cek_unik(raw["items"], ["order_id", "order_item_id"], "items")
cek_unik(raw["customers"], ["customer_id"], "customers")
cek_unik(raw["products"], ["product_id"], "products")
cek_unik(raw["category_tr"], ["product_category_name"], "category_tr")

print("\nDuplikat baris penuh:", {n: int(d.duplicated().sum()) for n, d in raw.items()})

orders       key=['order_id']: 0 duplikat  -> OK
items        key=['order_id', 'order_item_id']: 0 duplikat  -> OK
customers    key=['customer_id']: 0 duplikat  -> OK
products     key=['product_id']: 0 duplikat  -> OK
category_tr  key=['product_category_name']: 0 duplikat  -> OK



Duplikat baris penuh: {'orders': 0, 'items': 0, 'customers': 0, 'products': 0, 'category_tr': 0}


## 4. Jebakan customer_id vs customer_unique_id

Di Olist, `customer_id` dibuat **baru untuk setiap order**. Customer yang sama bisa punya beberapa `customer_id`. Identitas customer yang sebenarnya ada di `customer_unique_id`.

Jika salah memakai `customer_id`, semua customer akan terlihat sebagai customer baru, dan analisis new vs returning jadi salah total. Sel di bawah membuktikannya dengan angka.

In [4]:
cust = raw["customers"]
print("customer_id unik       :", f"{cust['customer_id'].nunique():,}")
print("customer_unique_id unik:", f"{cust['customer_unique_id'].nunique():,}")

customer_id unik       : 99,441
customer_unique_id unik: 96,096


## 5. Status order: mana yang dihitung?

**Keputusan scope (dari BRD):** revenue hanya dihitung dari order berstatus `delivered`. Order yang dibatalkan, belum dikirim, atau tidak tersedia tidak menjadi pendapatan yang benar-benar terealisasi.

In [5]:
orders = raw["orders"].copy()
print(orders["order_status"].value_counts().to_string())

date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

delivered = orders[orders["order_status"] == "delivered"].copy()
print(f"\nDelivered orders: {len(delivered):,} dari {len(orders):,} ({len(delivered)/len(orders):.1%})")
print("Delivered tapi tanggal diterima kosong:", int(delivered["order_delivered_customer_date"].isna().sum()))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2



Delivered orders: 96,478 dari 99,441 (97.0%)


Delivered tapi tanggal diterima kosong: 8


## 6. Cek bulan yang datanya tidak lengkap

Dataset biasanya dimulai dan diakhiri di tengah periode. Bulan yang datanya tidak lengkap membuat grafik tren dan angka growth menyesatkan (misalnya growth -95% padahal datanya saja yang terpotong).

**Aturan kelengkapan bulan.** Sebuah bulan hanya dipakai jika lolos ketiga cek berikut:

1. **Total bulanan** tidak jauh di bawah bulan-bulan di sekitarnya. Bulan yang benar-benar kosong ikut ditampilkan supaya tidak terlewat.
2. **Porsi order delivered** normal dibanding bulan lain, karena bulan terakhir dataset bisa berisi order yang belum selesai dikirim.
3. **Bulan terakhir jendela dicek per hari** (order semua status): rata-rata order harian di 7 hari terakhir bulan itu minimal 80% dari rata-rata hari-hari sebelumnya di bulan yang sama.

**Kenapa cek ketiga ditambahkan.** Pada keputusan awal, jendela analisis berakhir di Agustus 2018 karena total bulanan dan porsi delivered Agustus tampak normal. Pemeriksaan harian di Fase 5 kemudian menemukan bahwa order Agustus menyusut tajam setelah tanggal 21: rata-rata sekitar 277 order per hari pada 1–21 Agustus, tetapi hanya sekitar 70 per hari pada 22–31 Agustus. Pengumpulan data berakhir di tengah bulan, dan total bulanan menutupi masalah itu. Karena itu bulan terakhir wajib dicek sampai level harian, dan jendela analisis dipendekkan menjadi **2017-01 s/d 2018-07**.

Output di bawah menampilkan ketiga cek. Keputusannya dicatat di bagian *Catatan keputusan*.

In [6]:
delivered["order_month"] = delivered["order_purchase_timestamp"].dt.to_period("M")
monthly_orders = delivered.groupby("order_month").size()

# Cek 1 & 2: total order per bulan (semua status vs delivered) dan porsi delivered.
# Bulan yang benar-benar kosong ikut ditampilkan, supaya tidak terlewat.
order_month_all = orders["order_purchase_timestamp"].dt.to_period("M")
bulan = pd.period_range(order_month_all.min(), order_month_all.max(), freq="M")
cek_bulan = pd.DataFrame({
    "semua_status": order_month_all.value_counts().reindex(bulan, fill_value=0),
    "delivered": monthly_orders.reindex(bulan, fill_value=0),
})
cek_bulan["pct_delivered"] = (
    cek_bulan["delivered"] / cek_bulan["semua_status"].where(cek_bulan["semua_status"] > 0) * 100
).round(1)
print(cek_bulan.to_string())

# ---- Keputusan jendela analisis (diambil bersama user, lihat Catatan keputusan) ----
ANALYSIS_MONTHS = ("2017-01", "2018-07")

# Cek 3: bulan TERAKHIR jendela dicek per hari (semua status), dibandingkan dengan
# bulan sesudahnya. Tanda data terpotong: order harian menyusut di akhir bulan.
# Batas: rata-rata 7 hari terakhir minimal 80% dari rata-rata hari-hari sebelumnya.
akhir = pd.Period(ANALYSIS_MONTHS[1], freq="M")
harian = orders["order_purchase_timestamp"].dt.normalize().value_counts()
harian = harian.reindex(
    pd.date_range(akhir.start_time, (akhir + 1).end_time.normalize(), freq="D"), fill_value=0
)

print(f"\nCek harian (semua status): bulan terakhir jendela ({akhir}) dan bulan sesudahnya ({akhir + 1})")
for p in [akhir, akhir + 1]:
    h = harian[str(p)]
    for a in range(1, h.index.day.max() + 1, 7):
        blok = h[(h.index.day >= a) & (h.index.day < a + 7)]
        print(f"  {p} tgl {a:>2}-{blok.index.day.max():>2}: "
              f"rata-rata {blok.mean():6.1f} order/hari, terendah {blok.min():>3}")
    rasio = h.iloc[-7:].mean() / h.iloc[:-7].mean()
    status = "lengkap" if rasio >= 0.8 else "TERPOTONG"
    print(f"  -> rasio 7 hari terakhir {p} vs hari-hari sebelumnya: {rasio:.2f} ({status})\n")

         semua_status  delivered  pct_delivered
2016-09             4          1           25.0
2016-10           324        265           81.8
2016-11             0          0            NaN
2016-12             1          1          100.0
2017-01           800        750           93.8
2017-02          1780       1653           92.9
2017-03          2682       2546           94.9
2017-04          2404       2303           95.8
2017-05          3700       3546           95.8
2017-06          3245       3135           96.6
2017-07          4026       3872           96.2
2017-08          4331       4193           96.8
2017-09          4285       4150           96.8
2017-10          4631       4478           96.7
2017-11          7544       7289           96.6
2017-12          5673       5513           97.2
2018-01          7269       7069           97.2
2018-02          6728       6555           97.4
2018-03          7211       7003           97.1
2018-04          6939       6798        

## 7. Cek harga & ongkir (order items)
Mencari nilai yang tidak masuk akal: harga ≤ 0, ongkir negatif. Ongkir = 0 bisa wajar (gratis ongkir), jadi hanya dilaporkan, tidak dibuang.

In [7]:
items = raw["items"].copy()
print(items[["price", "freight_value"]].describe().round(2).to_string())
print("\nprice <= 0     :", int((items["price"] <= 0).sum()))
print("freight < 0    :", int((items["freight_value"] < 0).sum()))
print("freight == 0   :", int((items["freight_value"] == 0).sum()))

no_items = ~delivered["order_id"].isin(items["order_id"])
print("Delivered orders tanpa item:", int(no_items.sum()))

           price  freight_value
count  112650.00      112650.00
mean      120.65          19.99
std       183.63          15.81
min         0.85           0.00
25%        39.90          13.08
50%        74.99          16.26
75%       134.90          21.15
max      6735.00         409.68



price <= 0     : 0
freight < 0    : 0
freight == 0   : 383
Delivered orders tanpa item: 0


## 8. Produk & terjemahan kategori

Nama kategori asli berbahasa Portugis. Kita gabungkan dengan tabel terjemahan agar dashboard berbahasa Inggris.
- Produk tanpa kategori → diberi label `unknown` (tidak dibuang, supaya revenue tidak hilang).
- Kategori tanpa terjemahan → memakai nama Portugis aslinya.

In [8]:
products = (
    raw["products"][["product_id", "product_category_name"]]
    .merge(raw["category_tr"], on="product_category_name", how="left")
)

print("Produk tanpa kategori:", int(products["product_category_name"].isna().sum()))
no_tr = products.loc[
    products["product_category_name"].notna() & products["product_category_name_english"].isna(),
    "product_category_name",
].unique()
print("Kategori tanpa terjemahan:", list(no_tr))

products["category_pt"] = products["product_category_name"].fillna("unknown")
products["category_en"] = products["product_category_name_english"].fillna(products["category_pt"])
dim_product = products[["product_id", "category_en", "category_pt"]].copy()
print(f"\ndim_product: {len(dim_product):,} produk, {dim_product['category_en'].nunique()} kategori")

Produk tanpa kategori: 610
Kategori tanpa terjemahan: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

dim_product: 32,951 produk, 74 kategori


## 9. Membangun fact_sales

**Grain (1 baris = ...):** satu item dalam satu order yang delivered. Di Olist, satu baris order item = 1 unit produk, jadi quantity = jumlah baris.

Kolom:
- `price` = revenue item (ongkir **tidak** termasuk, sesuai definisi KPI)
- `freight_value` = ongkir yang **dibayar customer** (bukan biaya perusahaan/seller), dipakai untuk proxy *freight burden*
- `customer_state` = state customer **saat order itu dibuat**

In [9]:
fact = (
    items[["order_id", "order_item_id", "product_id", "price", "freight_value"]]
    .merge(delivered[["order_id", "customer_id", "order_purchase_timestamp"]], on="order_id", how="inner")
    .merge(raw["customers"][["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
)
fact["order_date"] = fact["order_purchase_timestamp"].dt.normalize()

fact_sales = fact[[
    "order_id", "order_item_id", "customer_unique_id", "product_id",
    "order_date", "customer_state", "price", "freight_value",
]].copy()

print(f"fact_sales: {len(fact_sales):,} baris, {fact_sales['order_id'].nunique():,} order")
print("customer_unique_id kosong:", int(fact_sales["customer_unique_id"].isna().sum()))

fact_sales: 110,197 baris, 96,478 order
customer_unique_id kosong: 0


## 10. dim_customer & dim_date

`dim_customer`: satu baris per customer unik, dengan **tanggal order pertama** (dipakai untuk menentukan customer baru vs returning) serta city/state dari order pertamanya.

`dim_date`: kalender harian **dari 1 Januari tahun pertama sampai 31 Desember tahun terakhir**, bukan hanya sepanjang rentang transaksi. Ini disengaja: fungsi waktu Power BI (`DATEADD`, `SAMEPERIODLASTYEAR`) hanya mengenal tanggal yang ada di tabel ini, jadi kalau kalendernya berhenti di tengah bulan, perbandingan bulan terakhir tidak akan memakai bulan pembanding yang penuh. Hari tambahan tidak punya transaksi sehingga tidak ada KPI yang berubah.

`dim_date` juga memuat kolom **`is_analysis_month`**, yaitu hasil keputusan di bagian 6 (`ANALYSIS_MONTHS`). Kolom ini menandai bulan yang datanya lengkap (**2017-01 s/d 2018-07**). Data bulan lain tetap dimuat ke database, tetapi ditandai `FALSE` supaya bisa dikecualikan lewat filter yang sama di SQL maupun Power BI. Dengan satu sumber penanda seperti ini, angka di notebook, SQL, dan dashboard tidak mungkin berbeda hanya karena filter yang lupa dipasang.

In [10]:
first_order = (
    fact.sort_values("order_date")
    .groupby("customer_unique_id", as_index=False)
    .first()[["customer_unique_id", "customer_city", "customer_state", "order_date"]]
    .rename(columns={"order_date": "first_order_date"})
)
dim_customer = first_order
print(f"dim_customer: {len(dim_customer):,} customer unik")

# Kalender dibuat TAHUN PENUH (1 Jan tahun pertama s/d 31 Des tahun terakhir),
# bukan hanya sepanjang rentang transaksi. Alasannya: fungsi waktu Power BI
# (DATEADD, SAMEPERIODLASTYEAR) hanya mengenal tanggal yang ada di tabel ini.
# Kalau kalender berhenti di tengah bulan, perbandingan MoM bulan terakhir tidak
# memakai bulan pembanding yang penuh. Hari tambahan tidak punya transaksi,
# jadi tidak ada satu pun KPI yang berubah.
first_year = fact_sales["order_date"].min().year
last_year = fact_sales["order_date"].max().year
dates = pd.date_range(f"{first_year}-01-01", f"{last_year}-12-31", freq="D")

dim_date = pd.DataFrame({"date_key": dates})
dim_date["year"] = dates.year
dim_date["quarter"] = dates.quarter
dim_date["month"] = dates.month
dim_date["month_name"] = dates.strftime("%b")
dim_date["year_month"] = dates.strftime("%Y-%m")
dim_date["day_of_week"] = dates.dayofweek + 1  # 1 = Senin

# --- Penanda jendela analisis (ANALYSIS_MONTHS diputuskan di bagian 6) ---
# Bulan di luar jendela datanya tidak lengkap:
#   2016-09 (1 order delivered), 2016-10 (265), 2016-11 (tidak ada order sama sekali),
#   2016-12 (1), 2018-08 (order harian menyusut tajam setelah tanggal 21 karena
#   pengumpulan data berakhir), serta 2018-09 & 2018-10 (hanya 20 order, nol delivered).
# Data di luar jendela TETAP dimuat ke database, hanya ditandai. Dengan begitu SQL dan
# Power BI memfilternya lewat satu kolom yang sama, sehingga angka di semua tool konsisten.
dim_date["is_analysis_month"] = dim_date["year_month"].between(*ANALYSIS_MONTHS)

print(f"dim_date: {len(dim_date):,} hari ({dates.min().date()} s/d {dates.max().date()})")
print(f"  ditandai bulan lengkap: {dim_date['is_analysis_month'].sum():,} hari "
      f"({ANALYSIS_MONTHS[0]} s/d {ANALYSIS_MONTHS[1]})")
print(f"  hari tanpa transaksi   : {len(dim_date) - dim_date['date_key'].isin(fact_sales['order_date']).sum():,} hari")

dim_customer: 93,358 customer unik
dim_date: 1,096 hari (2016-01-01 s/d 2018-12-31)
  ditandai bulan lengkap: 577 hari (2017-01 s/d 2018-07)
  hari tanpa transaksi   : 484 hari


## 11. Validasi sebelum export

Assert = pengecekan otomatis. Jika ada yang gagal, notebook berhenti dengan error. Ini lebih baik daripada data salah masuk ke dashboard tanpa ketahuan.

In [11]:
assert fact_sales.duplicated(["order_id", "order_item_id"]).sum() == 0, "Ada duplikat item"
assert fact_sales["customer_unique_id"].notna().all(), "Ada item tanpa customer"
assert fact_sales["product_id"].isin(dim_product["product_id"]).all(), "Ada produk tidak dikenal"
assert fact_sales["customer_unique_id"].isin(dim_customer["customer_unique_id"]).all()
assert fact_sales["order_date"].isin(dim_date["date_key"]).all()
assert dim_customer["customer_unique_id"].is_unique

expected_revenue = items.loc[items["order_id"].isin(delivered["order_id"]), "price"].sum()
assert abs(fact_sales["price"].sum() - expected_revenue) < 0.01, "Revenue berubah setelah join"
print("Semua validasi lolos.")


def kpi(df, judul):
    """Angka referensi. Harus SAMA PERSIS dengan hasil SQL dan Power BI."""
    revenue = df["price"].sum()
    n_orders = df["order_id"].nunique()
    print(f"\n--- {judul} ---")
    print(f"Total revenue   : {revenue:,.2f}")
    print(f"Total orders    : {n_orders:,}")
    print(f"Total customers : {df['customer_unique_id'].nunique():,}")
    print(f"Units sold      : {len(df):,}")
    print(f"AOV             : {revenue / n_orders:,.2f}")
    print(f"Freight burden  : {df['freight_value'].sum() / revenue:.2%}  (proxy: ongkir dibayar customer / harga barang; BUKAN profit, BUKAN biaya perusahaan)")


# (a) Seluruh data delivered — inilah yang dimuat ke database apa adanya.
kpi(fact_sales, "SEMUA data delivered (2016-09 s/d 2018-08)")

# (b) Jendela analisis: hanya bulan dengan is_analysis_month = TRUE.
# INI angka utama project. SQL dan Power BI memakai filter yang sama,
# jadi ketiga tool harus menghasilkan angka yang identik.
window_dates = dim_date.loc[dim_date["is_analysis_month"], "date_key"]
fact_window = fact_sales[fact_sales["order_date"].isin(window_dates)]
kpi(fact_window, f"JENDELA ANALISIS {ANALYSIS_MONTHS[0]} s/d {ANALYSIS_MONTHS[1]}")

n_all, n_win = fact_sales["order_id"].nunique(), fact_window["order_id"].nunique()
print(f"\nDikecualikan dari jendela: {n_all - n_win:,} order "
      f"({1 - n_win / n_all:.2%} dari seluruh order delivered)")

Semua validasi lolos.

--- SEMUA data delivered (2016-09 s/d 2018-08) ---
Total revenue   : 13,221,498.11
Total orders    : 96,478
Total customers : 93,358
Units sold      : 110,197
AOV             : 137.04
Freight burden  : 16.63%  (proxy: ongkir dibayar customer / harga barang; BUKAN profit, BUKAN biaya perusahaan)

--- JENDELA ANALISIS 2017-01 s/d 2018-07 ---
Total revenue   : 12,342,450.49
Total orders    : 89,860


Total customers : 86,960
Units sold      : 102,738
AOV             : 137.35
Freight burden  : 16.57%  (proxy: ongkir dibayar customer / harga barang; BUKAN profit, BUKAN biaya perusahaan)

Dikecualikan dari jendela: 6,618 order (6.86% dari seluruh order delivered)


## 12. Export

In [12]:
for name, df in {
    "fact_sales": fact_sales,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_date": dim_date,
}.items():
    df.to_csv(OUT_DIR / f"{name}.csv", index=False, date_format="%Y-%m-%d")
    print(f"Tersimpan: {name}.csv ({len(df):,} baris)")

Tersimpan: fact_sales.csv (110,197 baris)
Tersimpan: dim_customer.csv (93,358 baris)


Tersimpan: dim_product.csv (32,951 baris)
Tersimpan: dim_date.csv (1,096 baris)


## Catatan keputusan

Semua angka di bawah berasal dari output notebook ini dengan data Olist asli. Mata uang dataset adalah Real Brasil (R$).

1. **Order yang dipakai:** hanya `delivered`, yaitu **96.478 dari 99.441 order (97,0%)**. Sisanya shipped 1.107, canceled 625, unavailable 609, invoiced 314, processing 301, created 5, approved 2 — belum menjadi pendapatan yang terealisasi.
2. **Customer:** memakai `customer_unique_id`. Di tabel customers, **99.441 `customer_id` ternyata hanya mewakili 96.096 customer unik**, karena Olist membuat `customer_id` baru setiap order. Setelah dibatasi ke order delivered, `dim_customer` berisi **93.358 customer**.
3. **Jendela analisis 2017-01 s/d 2018-07.** Bulan yang dikecualikan karena datanya tidak lengkap (bagian 6):
   - 2016-09 (1 order delivered), 2016-10 (265), 2016-11 (**tidak ada order sama sekali**), 2016-12 (1).
   - **2018-08**: total bulanannya tampak normal (6.351 order delivered, porsi delivered 97,5%), tetapi **gagal cek harian** — order harian menyusut tajam setelah tanggal 21 (rasio 7 hari terakhir 0,15), tanda pengumpulan data berakhir di tengah bulan.
   - 2018-09 dan 2018-10: hanya 20 order, **nol** delivered.

   Januari 2017 (750 order) **tetap dipakai** karena porsi delivered-nya normal (93,8%) — kecilnya karena bisnisnya memang masih kecil, bukan karena datanya terpotong. Juli 2018, sebagai bulan terakhir jendela, **lolos cek harian** (rasio 1,25). Total yang dikecualikan: **6.618 dari 96.478 order delivered (6,86%)**. Diterapkan lewat kolom `dim_date.is_analysis_month`.

   **Kenapa ada cek harian:** jendela awalnya berakhir di Agustus 2018, karena cek total bulanan dan porsi delivered tidak menangkap bahwa data Agustus terpotong. Masalah itu baru terlihat di level harian (Fase 5), sehingga bulan terakhir jendela sekarang wajib dicek per hari.
4. **Produk tanpa kategori: 610 dari 32.951 (1,85%)** → diberi label `unknown`, tidak dibuang, supaya revenue produk tersebut tidak hilang dari laporan. Ada juga 2 kategori tanpa terjemahan Inggris (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) → memakai nama Portugis aslinya. Total menjadi **74 kategori**.
5. **Nilai harga/ongkir tidak wajar: tidak ada.** `price <= 0`: **0 baris**; `freight_value < 0`: **0 baris**. Ada **383 item dengan ongkir 0** (gratis ongkir) — wajar secara bisnis, jadi dipertahankan. Pada seluruh order item: harga R$ 0,85–6.735,00 dengan median R$ 74,99 (rata-rata R$ 120,65, jadi distribusinya miring ke kanan).
6. **Order delivered tanpa item: 0.** Jumlah order di `fact_sales` (96.478) karena itu persis sama dengan jumlah order delivered, sehingga definisi Total Orders tidak ambigu.
7. **8 order delivered tanpa tanggal diterima** — tidak berpengaruh pada analisis, karena sumbu waktu memakai tanggal **pembelian**, bukan tanggal terima.

### Angka referensi

Jendela analisis 2017-01 s/d 2018-07. Hasil SQL (`sql/05_kpi_reference.sql`) dan Power BI **harus identik** dengan ini:

| KPI | Nilai |
|---|---|
| Total Revenue | R$ 12.342.450,49 |
| Total Orders | 89.860 |
| Total Customers | 86.960 |
| Units Sold | 102.738 |
| AOV | R$ 137,35 |
| Freight Burden (proxy: ongkir dibayar customer ÷ harga barang; bukan profit, bukan biaya perusahaan) | 16,57% |

Sebagai pembanding, seluruh data delivered (termasuk bulan tidak lengkap): revenue R$ 13.221.498,11 · 96.478 order · 93.358 customer · 110.197 unit · AOV R$ 137,04.